<a href="https://colab.research.google.com/github/zesia08/magic-seller/blob/main/magic_seller_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [STEP 1] 라이브러리 설치 (이름 수정!)
!pip install supabase pandas -q

# [STEP 2] 임포트
from supabase import create_client
import pandas as pd
from datetime import datetime

# [STEP 3] 본인 Supabase 정보 입력
SUPABASE_URL = "https://hhhygjtrmocbpedjcfat.supabase.co"  #
SUPABASE_KEY = "sb_publishable_EZJTSgPEtauD-_cC-BHgZQ__nAtSrQR"  #

# [STEP 4] 연결 테스트
try:
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("[OK] Supabase 연결 성공!")

    # 테이블 목록 확인
    response = supabase.table('companies').select("*").limit(3).execute()
    print(f"[OK] companies 테이블: {len(response.data)}건 조회됨")

    # 인플루언서도 확인
    inf_response = supabase.table('influencers').select("*").limit(3).execute()
    print(f"[OK] influencers 테이블: {len(inf_response.data)}건 조회됨")

except Exception as e:
    print(f"[ERROR] {e}")

# [STEP 5] AI 매칭 엔진
class AIMatcher:
    def __init__(self):
        self.weights = {
            'category': 20,
            'price': 15,
            'recency': 15,
            'korea': 15,
            'audience': 10,
            'response': 10,
            'sales': 10,
            'growth': 5
        }

    def calculate(self, company, influencer):
        scores = {}

        # 1. 카테고리
        scores['category'] = self.weights['category'] if company.get('category') == influencer.get('category') else 0

        # 2. 가격대
        price_diff = abs(company.get('target_price', 0) - influencer.get('avg_price', 0))
        scores['price'] = max(0, self.weights['price'] - (price_diff / 10000))

        # 3. 최근 활동
        days = influencer.get('days_since_live', 30)
        if days <= 3: scores['recency'] = 15
        elif days <= 7: scores['recency'] = 10
        elif days <= 30: scores['recency'] = 5
        else: scores['recency'] = 0

        # 4. 한국제품
        scores['korea'] = 15 if influencer.get('korean_count', 0) > 0 else 0

        # 5. 타겟
        scores['audience'] = influencer.get('target_match', 0) * 10

        # 6. 응답률
        scores['response'] = influencer.get('response_rate', 0) * 10

        # 7. 매출
        scores['sales'] = min(10, influencer.get('monthly_sales', 0) / 50000)

        # 8. 성장
        growth = influencer.get('growth', 0)
        scores['growth'] = 5 if growth > 0.2 else 2 if growth > 0 else 0

        total = sum(scores.values())
        grade = 'S' if total >= 90 else 'A' if total >= 80 else 'B' if total >= 70 else 'C'

        return {
            'total': min(100, int(total)),
            'grade': grade,
            'details': scores
        }

# [STEP 6] 실제 DB 데이터로 AI 매칭 실행
print("\n" + "="*60)
print("[실제 DB 데이터로 AI 매칭 테스트]")
print("="*60)

# 기업 데이터 (새로운 기업이라고 가정)
new_company = {
    'name': '한국 화장품 브랜드',
    'category': 'beauty',
    'target_price': 50000,
    'target_age': '20s_female'
}

# DB에서 인플루언서 가져와서 매칭
try:
    response = supabase.table('influencers').select("*").execute()
    influencers = response.data

    if not influencers:
        print("[INFO] DB에 인플루언서 데이터가 없습니다. 샘플로 테스트합니다.")
        # 샘플 데이터
        influencers = [
            {'name': '小美Beauty', 'category': 'beauty', 'avg_price': 48000, 'days_since_live': 2, 'korean_count': 5, 'target_match': 0.85, 'response_rate': 0.9, 'monthly_sales': 300000, 'growth': 0.25},
            {'name': '时尚达人Lily', 'category': 'fashion', 'avg_price': 60000, 'days_since_live': 5, 'korean_count': 0, 'target_match': 0.60, 'response_rate': 0.7, 'monthly_sales': 150000, 'growth': 0.10},
            {'name': '美妆教主Coco', 'category': 'beauty', 'avg_price': 55000, 'days_since_live': 3, 'korean_count': 8, 'target_match': 0.90, 'response_rate': 0.85, 'monthly_sales': 450000, 'growth': 0.15}
        ]

    # AI 매칭 실행
    matcher = AIMatcher()
    results = []

    for inf in influencers:
        match = matcher.calculate(new_company, inf)
        results.append({
            'influencer': inf,
            'match': match
        })

    # 점수 높은 순 정렬
    results.sort(key=lambda x: x['match']['total'], reverse=True)

    # TOP 5 출력
    print(f"\n기업: {new_company['name']}")
    print(f"카테고리: {new_company['category']} / 희망가격: ₩{new_company['target_price']:,}\n")

    for i, r in enumerate(results[:5], 1):
        inf = r['influencer']
        m = r['match']

        print(f"[{i}] Grade {m['grade']} | {m['total']}점 | {inf['name']}")
        print(f"    팔로워: {(inf.get('followers', 0)/10000):.1f}만 | 매출: ₩{inf.get('monthly_sales', 0):,}")
        print(f"    강점: ", end="")

        strengths = []
        if m['details']['category'] == 20: strengths.append("카테고리완벽")
        if m['details']['korea'] == 15: strengths.append("한국제품경험")
        if m['details']['recency'] >= 10: strengths.append("최근활동")
        if m['details']['total'] >= 80: strengths.append("고점수")

        print(", ".join(strengths) if strengths else "기본매칭")
        print()

except Exception as e:
    print(f"[ERROR] 매칭 실행 실패: {e}")

print("[완료] AI 매칭 시스템 정상 작동")

In [ ]:
from IPython.display import HTML, display

# HTML 코드를 문자열로 감싸기
html_content = """
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>매직셀러 - 크레딧 충전</title>
    <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-50 font-sans">
    <div class="container mx-auto mt-8 px-4 max-w-2xl">

        <!-- 결제 섹션 추가 -->
        <div class="bg-white p-8 rounded-lg shadow">
            <h3 class="text-xl font-bold mb-4">크레딧 충전</h3>

            <div class="grid grid-cols-3 gap-4">
                <button onclick="charge(10000)" class="p-4 border rounded hover:bg-blue-50 text-center">
                    <div class="text-2xl font-bold text-blue-600">100C</div>
                    <div class="text-gray-600">10,000원</div>
                </button>
                <button onclick="charge(40000)" class="p-4 border rounded hover:bg-blue-50 text-center border-blue-500 bg-blue-50">
                    <div class="text-2xl font-bold text-blue-600">400C</div>
                    <div class="text-gray-600">40,000원</div>
                    <div class="text-xs text-red-500">20% 할인</div>
                </button>
                <button onclick="charge(70000)" class="p-4 border rounded hover:bg-blue-50 text-center">
                    <div class="text-2xl font-bold text-blue-600">700C</div>
                    <div class="text-gray-600">70,000원</div>
                    <div class="text-xs text-red-500">30% 할인</div>
                </button>
            </div>
        </div>

        <!-- 토스 결제 SDK -->
        <script src="https://js.tosspayments.com/v1/payment"></script>

        <script>
            // 토스 테스트 클라이언트 키 (실제로는 내 키로 변경)
            const clientKey = 'test_ck_E92LAa5PVbP01p91MPaR87WlpXyJ';
            const tossPayments = TossPayments(clientKey);

            function charge(amount) {
                const orderId = 'order_' + Date.now();

                tossPayments.requestPayment('카드', {
                    amount: amount,
                    orderId: orderId,
                    orderName: '매직셀러 크레딧 ' + (amount/100) + 'C 충전',
                    customerName: '테스트 사용자',
                    successUrl: window.location.origin + '/success.html',
                    failUrl: window.location.origin + '/fail.html'
                });
            }
        </script>
    </div>
</body>
</html>
"""

# HTML 렌더링
display(HTML(html_content))
print("[OK] HTML 렌더링 완료! 위에 크레딧 충전 UI가 표시됩니다.")

In [ ]:
from IPython.display import HTML, display

success_html = """
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>결제 성공</title>
    <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-green-50 flex items-center justify-center h-screen">
    <div class="bg-white p-8 rounded-lg shadow text-center">
        <div class="text-6xl mb-4">SUCCESS</div>  <!-- 여기! 🎉 → SUCCESS -->
        <h1 class="text-2xl font-bold text-green-600 mb-4">결제 성공!</h1>
        <p class="mb-4">크레딧이 자동으로 충전되었습니다.</p>
        <a href="index.html" class="bg-green-600 text-white px-6 py-2 rounded hover:bg-green-700">
            메인으로 돌아가기
        </a>
    </div>
</body>
</html>
"""

display(HTML(success_html))
print("[OK] 결제 성공 페이지 렌더링 완료!")

In [ ]:
from IPython.display import HTML, display

fail_html = """
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>결제 실패</title>
    <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-red-50 flex items-center justify-center h-screen">
    <div class="bg-white p-8 rounded-lg shadow text-center">
        <div class="text-6xl mb-4">FAILED</div>  <!-- 여기! 😢 → FAILED -->
        <h1 class="text-2xl font-bold text-red-600 mb-4">결제 실패</h1>
        <p class="mb-4">다시 시도해 주세요.</p>
        <a href="index.html" class="bg-red-600 text-white px-6 py-2 rounded hover:bg-red-700">
            다시 시도
        </a>
    </div>
</body>
</html>
"""

display(HTML(fail_html))
print("[OK] 결제 실패 페이지 렌더링 완료!")

In [ ]:
import requests
import json
from datetime import datetime, timedelta
from supabase import create_client
import time

# 설정
SUPABASE_URL = "https://hhhygjtrmocbpedjcfat.supabase.co"
SUPABASE_KEY = "sb_publishable_EZJTSgPEtauD-_cC-BHgZQ__nAtSrQR"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def fetch_douyin_data(influencer_id):
    """
    실제 Douyin API 호출 (또는 웹 스크래핑)
    여기서는 mock 데이터로 시뮬레이션
    """
    return {
        'id': influencer_id,
        'last_live': datetime.now() - timedelta(days=2),
        'monthly_sales': 350000 + (hash(influencer_id) % 200000),
        'viewer_count': 15000 + (hash(influencer_id) % 10000),
        'response_rate': 0.85,
        'growth_rate': 0.15
    }

def update_influencer_stats():
    """모든 인플루언서 데이터 업데이트"""
    print(f"🤖 {datetime.now()} - 데이터 수집 시작")

    # DB에서 모든 인플루언서 가져오기
    response = supabase.table('influencers').select('id').execute()
    influencers = response.data

    for inf in influencers:
        try:
            # 새 데이터 수집
            new_data = fetch_douyin_data(inf['id'])

            # DB 업데이트
            supabase.table('influencers').update({
                'days_since_live': (datetime.now() - new_data['last_live']).days,
                'monthly_sales': new_data['monthly_sales'],
                'live_viewers': new_data['viewer_count'],
                'response_rate': new_data['response_rate'],
                'growth': new_data['growth_rate']
            }).eq('id', inf['id']).execute()

            print(f"  ✓ {inf['id']} 업데이트 완료")
            time.sleep(1)  # API 제한 준수

        except Exception as e:
            print(f"  ✗ {inf['id']} 오류: {e}")

    print(f"✅ {len(influencers)}명 업데이트 완료")

# 6시간마다 실행
if __name__ == "__main__":
    while True:
        update_influencer_stats()
        print("😴 6시간 대기 중...")
        time.sleep(6 * 3600)

🤖 2026-04-10 16:45:23.718169 - 데이터 수집 시작
✅ 0명 업데이트 완료
😴 6시간 대기 중...


In [ ]:
# notification.py
import smtplib
from email.mime.text import MIMEText
from supabase import create_client

def send_match_alert(company_email, top_matches):
    """기업에게 AI 매칭 결과 이메일 발송"""

    msg_content = f"""
    <h2>🎯 매직셀러 AI 매칭 결과</h2>
    <p>귀사의 제품에 가장 적합한 인플루언서 TOP 3를 추천드립니다:</p>
    <ol>
        <li><strong>{top_matches[0]['name']}</strong> - 적합도 {top_matches[0]['score']}%</li>
        <li><strong>{top_matches[1]['name']}</strong> - 적합도 {top_matches[1]['score']}%</li>
        <li><strong>{top_matches[2]['name']}</strong> - 적합도 {top_matches[2]['score']}%</li>
    </ol>
    <p><a href="https://magic-seller.vercel.app">자세히 보기</a></p>
    """

    msg = MIMEText(msg_content, 'html')
    msg['Subject'] = '[매직셀러] AI 인플루언서 매칭 완료'
    msg['From'] = 'noreply@magic-seller.com'
    msg['To'] = company_email

    # Gmail SMTP (앱 비밀번호 사용)
    with smtplib.SMTP('smtp.gmail.com', 587) as server:
        server.starttls()
        server.login('zesia08@gmail.com', 'lms1667715~~')
        server.send_message(msg)

    print(f"📧 {company_email}로 알림 발송 완료")

In [ ]:
from IPython.display import HTML, display

admin_html = """
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>매직셀러 운영자 대시보드</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>
<body class="bg-gray-100">
    <div class="container mx-auto p-6">
        <h1 class="text-3xl font-bold mb-6">[ADMIN] 매직셀러 운영자 대시보드</h1>

        <!-- 핵심 지표 -->
        <div class="grid grid-cols-4 gap-4 mb-6">
            <div class="bg-white p-4 rounded-lg shadow">
                <div class="text-gray-500 text-sm">오늘 가입 기업</div>
                <div class="text-3xl font-bold text-blue-600" id="todaySignups">0</div>
            </div>
            <div class="bg-white p-4 rounded-lg shadow">
                <div class="text-gray-500 text-sm">오늘 매칭 요청</div>
                <div class="text-3xl font-bold text-purple-600" id="todayMatches">0</div>
            </div>
            <div class="bg-white p-4 rounded-lg shadow">
                <div class="text-gray-500 text-sm">오늘 결제 금액</div>
                <div class="text-3xl font-bold text-green-600" id="todayRevenue">0원</div>
            </div>
            <div class="bg-white p-4 rounded-lg shadow">
                <div class="text-gray-500 text-sm">미처리 알림</div>
                <div class="text-3xl font-bold text-red-600" id="pendingAlerts">0</div>
            </div>
        </div>

        <!-- 자동화 상태 -->
        <div class="bg-white rounded-lg shadow p-6 mb-6">
            <h2 class="text-xl font-bold mb-4">[BOT] 자동화 시스템 상태</h2>
            <div class="space-y-3">
                <div class="flex justify-between items-center p-3 bg-green-50 rounded">
                    <span>데이터 수집 봇</span>
                    <span class="text-green-600 font-bold">[ON] 정상 작동 중</span>
                </div>
                <div class="flex justify-between items-center p-3 bg-green-50 rounded">
                    <span>AI 매칭 엔진</span>
                    <span class="text-green-600 font-bold">[ON] 정상 작동 중</span>
                </div>
                <div class="flex justify-between items-center p-3 bg-green-50 rounded">
                    <span>결제 시스템</span>
                    <span class="text-green-600 font-bold">[ON] 정상 작동 중</span>
                </div>
                <div class="flex justify-between items-center p-3 bg-yellow-50 rounded">
                    <span>이메일 알림</span>
                    <span class="text-yellow-600 font-bold">[WAIT] 대기열 3건</span>
                </div>
            </div>
        </div>

        <!-- 이상 징후 알림 -->
        <div class="bg-white rounded-lg shadow p-6">
            <h2 class="text-xl font-bold mb-4 text-red-600">[ALERT] 확인 필요 (수동 개입)</h2>
            <div id="alertList" class="space-y-2">
                <p class="text-gray-500">현재 이상 징후 없음 - 시스템이 자동으로 처리 중입니다</p>
            </div>
        </div>
    </div>

    <script>
        // Supabase 연동으로 실시간 데이터 표시
        // 30분마다 자동 새로고침
        setInterval(() => location.reload(), 30 * 60 * 1000);
    </script>
</body>
</html>
"""

display(HTML(admin_html))
print("[OK] 운영자 대시보드 렌더링 완료!")